# Halide Perovskite — 最小二乘 QUBO + CIM
## 图半监督学习: 材料相似图 → 图拉普拉斯 → $\min\|A x - b\|^2$ → CIM
36维元素描述符 → k-NN图 → 能量泛函系统 → LS-QUBO → CIM

In [2]:
import numpy as np
import pandas as pd
import kaiwu as kw
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings, json, os
warnings.filterwarnings('ignore')
kw.common.CheckpointManager.save_dir = '/tmp'

BIT_WIDTH = 8
K_NN = 8
LAMBDA_LABEL = 10.0
EPS_REG = 0.01
N_SUBGRAPH = 60
N_LABELED = 10
BOUND_MARGIN = 0.5

OUTPUT_DIR = 'D:/QPDE/photo+kan/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('模块加载完成')

模块加载完成


In [3]:
# ===== 加载数据 =====
url = 'https://raw.githubusercontent.com/mannodiarun/halide_perovs_design/main/PBE_data.csv'
df = pd.read_csv(url, header=None)
target_raw = df.iloc[:, 3].values.astype(float)
features_raw = df.iloc[:, 22:58].values.astype(float)
print(f'全部: {df.shape[0]} 条, 特征: {features_raw.shape[1]} 维')
print(f'PBE_gap 范围: [{target_raw.min():.4f}, {target_raw.max():.4f}]')

全部: 550 条, 特征: 36 维
PBE_gap 范围: [0.2688, 5.3672]


In [4]:
# ===== 特征标准化 + 采样子图 =====
scaler = StandardScaler().fit(features_raw)
features = scaler.transform(features_raw)

rng = np.random.default_rng(42)
idx_sample = rng.choice(len(features), N_SUBGRAPH, replace=False)
feat_sub = features[idx_sample]
target_sub = target_raw[idx_sample]
print(f'子图: {N_SUBGRAPH} 个材料')

子图: 60 个材料


In [5]:
# ===== 构建 k-NN 相似图 + 图拉普拉斯 =====
sim = cosine_similarity(feat_sub)
np.fill_diagonal(sim, -np.inf)
W = np.zeros((N_SUBGRAPH, N_SUBGRAPH))
for i in range(N_SUBGRAPH):
    top_k = np.argpartition(-sim[i], K_NN)[:K_NN]
    W[i, top_k] = sim[i, top_k]
W = (W + W.T) / 2

D_vec = W.sum(axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(D_vec, 1e-12)))
L_sym = np.eye(N_SUBGRAPH) - D_inv_sqrt @ W @ D_inv_sqrt
L_sym = (L_sym + L_sym.T) / 2
print(f'图: 边数={(W>0).sum()}')

图: 边数=566


In [6]:
# ===== 标记节点 & 构建系统 A x = b =====
labeled_idx = rng.choice(N_SUBGRAPH, N_LABELED, replace=False)
labeled_mask = np.zeros(N_SUBGRAPH, dtype=bool)
labeled_mask[labeled_idx] = True

# A = L_sym + λ I_L + ε I,  b_i = λ*y_i if labeled else 0
I_L = np.diag(labeled_mask.astype(float))
A_mat = L_sym + LAMBDA_LABEL * I_L + EPS_REG * np.eye(N_SUBGRAPH)
b_vec = LAMBDA_LABEL * labeled_mask.astype(float) * target_sub

print(f'A: {A_mat.shape}, κ(A)={np.linalg.cond(A_mat):.2e}')

A: (60, 60), κ(A)=1.57e+02


In [7]:
# ===== 经典参考解 & 编码边界 =====
x_ref = np.linalg.solve(A_mat, b_vec)
margin = BOUND_MARGIN
rng_val = max(x_ref.max() - x_ref.min(), 0.1)
lb = x_ref.min() - margin * rng_val
ub = x_ref.max() + margin * rng_val
print(f'参考解: [{x_ref.min():.4f}, {x_ref.max():.4f}], 界: [{lb:.4f}, {ub:.4f}]')

lb_vec = np.full(N_SUBGRAPH, lb)
ub_vec = np.full(N_SUBGRAPH, ub)
print(f'QUBO 变量: {N_SUBGRAPH}, 量子比特: {N_SUBGRAPH * BIT_WIDTH}')

参考解: [0.7336, 4.2923], 界: [-1.0458, 6.0717]
QUBO 变量: 60, 量子比特: 480


In [8]:
# ===== Binary Encoding =====
def build_binary_encoding(n_vars, bit_width, lower, upper):
    n_binary = n_vars * bit_width
    E = np.zeros((n_vars, n_binary), dtype=float)
    c = lower.copy().astype(float)
    denom = (2**bit_width) - 1.0
    for i in range(n_vars):
        step = (upper[i] - lower[i]) / denom
        for j in range(bit_width):
            E[i, i*bit_width + j] = step * (2**j)
    return E, c

def decode_binary_solution(z_binary, E, c):
    return E @ z_binary + c

E_mat, c_vec = build_binary_encoding(N_SUBGRAPH, BIT_WIDTH, lb_vec, ub_vec)
print(f'E: {E_mat.shape}, c: {c_vec.shape}')
print(f'二进制变量: {N_SUBGRAPH * BIT_WIDTH}')

E: (60, 480), c: (60,)
二进制变量: 480


In [9]:
# ===== 构建 LS-QUBO =====
# min ||A x - b||^2, x = E z + c
# ||A(Ez+c) - b||^2 = z^T E^T A^T A E z + 2 (Ac-b)^T A E z + const

def build_ls_qubo(A, b, E, c):
    AE = A @ E
    d = A @ c - b
    Q = AE.T @ AE
    Q = (Q + Q.T) / 2  # 对称化
    linear = 2.0 * (d @ AE)
    np.fill_diagonal(Q, Q.diagonal() + linear)
    return Q.astype(np.float32)

Q_float = build_ls_qubo(A_mat, b_vec, E_mat, c_vec)
print(f'原始 QUBO: {Q_float.shape}, 值: [{Q_float.min():.4f}, {Q_float.max():.4f}]')

Q_qubo = kw.qubo.adjust_qubo_matrix_precision(Q_float)
print(f'调整后 QUBO: [{Q_qubo.min():.4f}, {Q_qubo.max():.4f}]')

原始 QUBO: (480, 480), 值: [-2610.4368, 774.6484]
调整后 QUBO: [-916.0000, 536.0000]


In [14]:
# ===== 提交 CIM 任务 =====
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
n_vars = ising_mat.shape[0]
variables = [f'x[{i}]' for i in range(n_vars)]
ising_model = kw.ising.IsingModel(variables=variables, ising_matrix=ising_mat, bias=ising_bias)

print(f'Ising: {ising_mat.shape}, 变量数: {n_vars}')

optimizer = kw.cim.CIMOptimizer(task_name='perov_1ls', task_mode='quota')
optimizer.solve(ising_model.get_matrix())
print('CIM 任务已提交 (perov_ls)，等完成后跑下一个 cell')

Ising: (481, 481), 变量数: 481
[2026-05-20 18:20:49] [INFO    ] [kaiwu.cim._optimizer_adapter:10] - Task submit successfully, waiting for data validation. Task name: perov_1ls
CIM 任务已提交 (perov_ls)，等完成后跑下一个 cell


In [15]:
# ===== 取回 CIM + 解码 =====
sol = optimizer.solve(ising_model.get_matrix())
print(f'CIM 返回: {sol.shape}')

solutions = sol[:, :-1]
deltas = sol[:, -1]
solutions_binary = (solutions * deltas[:, np.newaxis] + 1) / 2

energies = np.array([s @ Q_qubo @ s for s in solutions_binary])
best_idx = np.argmin(energies)
z_best = solutions_binary[best_idx]

x_quantum = decode_binary_solution(z_best, E_mat, c_vec)

print(f'最佳解索引: {best_idx}, 能量: {energies[best_idx]:.6f}')
print(f'量子解: [{x_quantum.min():.4f}, {x_quantum.max():.4f}]')
rmse_all = np.sqrt(np.mean((x_quantum - x_ref)**2))
print(f'RMSE vs 参考: {rmse_all:.6e}')

[2026-05-20 18:21:44] [INFO    ] [kaiwu.cim._optimizer_adapter:2] - Task completed: perov_1ls
CIM 返回: (10, 481)
最佳解索引: 0, 能量: -6208.000000
量子解: [0.6568, 5.5134]
RMSE vs 参考: 1.299133e+00


In [16]:
# ===== 保存预设 =====
np.save(f'{OUTPUT_DIR}/perov_nodes_ls.npy', idx_sample)
np.save(f'{OUTPUT_DIR}/perov_features_ls.npy', feat_sub)
np.save(f'{OUTPUT_DIR}/perov_values_ls.npy', x_quantum)
np.save(f'{OUTPUT_DIR}/perov_target_ls.npy', target_sub)
np.save(f'{OUTPUT_DIR}/perov_labeled_mask_ls.npy', labeled_mask)
np.save(f'{OUTPUT_DIR}/perov_W_ls.npy', W)

rmse_vs_target = np.sqrt(np.mean((x_quantum - target_sub)**2))
meta = {
    'method': 'ls_graph',
    'n_materials': int(N_SUBGRAPH),
    'n_labeled': int(N_LABELED),
    'bit_width': BIT_WIDTH,
    'qubo_size': int(n_vars),
    'best_energy': float(energies[best_idx]),
    'rmse_vs_ref': float(rmse_all),
    'rmse_vs_target': float(rmse_vs_target),
    'kappa_A': float(np.linalg.cond(A_mat))
}
with open(f'{OUTPUT_DIR}/meta_ls.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'预设已保存, RMSE vs 参考={rmse_all:.6e}, RMSE vs DFT={rmse_vs_target:.6e}')

预设已保存, RMSE vs 参考=1.299133e+00, RMSE vs DFT=1.700252e+00
